# ColabUMA Scan — relaxed scan with UMA

Runs a **relaxed scan** with Meta's machine learning potential **UMA**: a bond length, angle or dihedral is moved in fixed steps while the other degrees of freedom are optimized. It corresponds to the `S` entry of Gaussian's `opt=modredundant`. No coding needed.

## How to use

1. In **Runtime → Change runtime type**, choose **T4 GPU**
2. First time only: open 🔑 (Secrets) in the left sidebar, add your Hugging Face access token with the name `HF_TOKEN`, and turn on notebook access
3. In **1. Input form** below, set the output name, charge, spin multiplicity, scan settings and convergence
4. Choose **Runtime → Run all**
5. A file selection button appears in **2. File selection and checks**. Choose an xyz or a Gaussian gjf / com file. You can leave it running after that
6. Check the energy profile and the structure of each point in the last cell, and download the ZIP of results

If you don't know the atom numbers, just run it once: "5. Prepare the structures" shows a numbered 3D view. Check the numbers, fix the form and run again. In the file selection, Cancel upload reuses the previous file.

To run again, repeat steps 3–6 (the install and the model load are skipped).

> **Before you start**: the model weights, [facebook/UMA](https://huggingface.co/facebook/UMA), require an access request. Log in to Hugging Face, accept the FAIR Chemistry License v1 on that page, and once approved, create a token with Read access under [Access Tokens](https://huggingface.co/settings/tokens).

## Scan settings

| Item | Description |
|---|---|
| `scan_atoms` | Atom numbers (from 1), separated by spaces. 2 atoms give a bond length, 3 an angle, 4 a dihedral |
| `scan_start` | Start value (Å or deg). Empty starts from the value in the input structure |
| `scan_n_steps` | Number of steps. The number of points is this plus 1 (as with Gaussian's `S`) |
| `scan_step_size` | Size of one step (Å or deg). A negative value moves the other way |
| `opt_before_scan` | If on, the input structure is optimized without constraints before the scan. With an empty `scan_start`, the scan starts from the optimized value |
| `opt_after_scan` | If on, a structure is optimized without constraints after the scan. It starts from the first minimum found going back from the last point |

For example, Gaussian's `D 1 2 3 4 S 36 10.0` is `scan_atoms = 1 2 3 4`, `scan_n_steps = 36`, `scan_step_size = 10`.

The post-scan optimization is for reaching the product-side minimum from a scan that went too far. Even if the energy rises because a bond was shortened too much, it goes back down that slope and optimizes from the minimum there. Bumps smaller than 0.1 kcal/mol and unconverged points are ignored.

## Output (contents of the ZIP)

| File | Contents |
|---|---|
| `<output name>_scan.csv` | Target and actual value, energy, ΔE, fmax, steps and convergence of each point (unconverged points included) |
| `<output name>_scan.xyz` | Optimized structure of each converged point (multi-structure xyz) |
| `<output name>_scan.log` | Final structures of the converged points in the format of Gaussian's scan output. Not a Gaussian calculation |
| `<output name>_preopt.xyz` / `.log` | Result and path of the pre-scan optimization (if on) |
| `<output name>_postopt.xyz` / `.log` | Result and path of the post-scan optimization (if on) |
| `<output name>_log.txt` | Input settings, software versions, energy and fmax of each point and step |

## Notes

- Each point starts from the optimized structure of the previous point. If the structure jumps to another valley, the profile shows a step. Scan in the opposite direction too and check that you get the same curve.
- The highest point of the profile is not a transition state. Use it as a candidate initial guess and optimize it separately.
- The charge and spin multiplicity are fixed for all points. In scans that break a bond, the energy near dissociation may not be correct.
- When the scanned coordinate is in a ring, only the end atom is moved, so the starting structures can be strongly distorted. Use a smaller step size.
- UMA is a machine learning potential, not a replacement for DFT. Check final energy comparisons with DFT.
- Periodic systems (crystals, surfaces) are not supported.
- To search for a transition state from the scan maximum or the post-scan structure, [ColabUMA_Workflow](https://colab.research.google.com/github/RChibaChiba/ColabChem/blob/main/ColabUMA_Workflow_en.ipynb) continues from the scan to the TS in the same session.

## License

- This notebook and the shared code (`colabuma`): [MIT License](https://github.com/RChibaChiba/ColabChem/blob/main/LICENSE)
- UMA model weights: [FAIR Chemistry License v1](https://huggingface.co/facebook/UMA)
- Libraries installed at runtime are used under their own licenses. See [License in the README](https://github.com/RChibaChiba/ColabChem/blob/main/README.md#license)

## Citation

If you use this notebook in a publication, please cite:

- B. M. Wood *et al.*, "UMA: A Family of Universal Models for Atoms", arXiv:2506.23971 (2025).
- D. S. Levine *et al.*, "The Open Molecules 2025 (OMol25) Dataset, Evaluations, and Models", arXiv:2505.08762 (2025).
- fairchem: https://github.com/facebookresearch/fairchem

In [ ]:
#@title 1. Input form { display-mode: "form" }
#@markdown When the form is filled in, choose **Runtime → Run all**. Pick an xyz or gjf file in the next cell; after that you can leave it running.

#@markdown ### Output
output_name = "result"  #@param {type:"string"}

#@markdown ### Structure
charge = 0  #@param {type:"integer"}
multiplicity = 1  #@param {type:"integer"}

#@markdown ### Scan settings
#@markdown Atom numbers (from 1), separated by spaces. 2 atoms give a bond length, 3 an angle, 4 a dihedral.
scan_atoms = "1 2 3 4"  #@param {type:"string"}
#@markdown Start value (Å or deg). Empty starts from the input structure (or from the optimized one if the pre-scan optimization is on).
scan_start = ""  #@param {type:"string"}
#@markdown Number of steps and step size (Å or deg). The number of points is the number of steps plus 1.
scan_n_steps = 36  #@param {type:"integer"}
scan_step_size = 10.0  #@param {type:"number"}
#@markdown Optimize the input structure without constraints before the scan.
opt_before_scan = True  #@param {type:"boolean"}
#@markdown After the scan, go back from the last point to the first minimum and optimize it without constraints.
opt_after_scan = True  #@param {type:"boolean"}

#@markdown ### Settings
model_name = "uma-s-1p1"  #@param ["uma-s-1p1", "uma-s-1p2", "uma-m-1p1"]
task_name = "omol"  #@param ["omol"]
optimizer_name = "LBFGS"  #@param ["LBFGS", "BFGS", "FIRE"]
#@markdown Convergence threshold fmax (eV/Å). normal is close to Gaussian's default, tight to Gaussian's opt=tight.
convergence = "normal (0.02)"  #@param ["loose (0.05)", "normal (0.02)", "tight (0.001)", "custom"]
#@markdown fmax (eV/Å) used when custom is chosen.
fmax_custom = 0.02  #@param {type:"number"}
#@markdown The maximum number of steps is per point.
max_steps = 500  #@param {type:"integer"}

In [ ]:
#@title 2. File selection and checks { display-mode: "form" }
#@markdown File selection and the token check happen here. You can leave the install and calculations after this running.

import os
import subprocess
import sys

COLABUMA_REF = "main"  # GitHub branch or tag of the shared code (colabuma package)


def _use_colabuma(ref):
    # Use colabuma next to the notebook if it is there, otherwise fetch it from GitHub
    if os.path.isdir("colabuma"):
        path = os.getcwd()
    else:
        path = "/content/ColabChem" if os.path.isdir("/content") else os.path.expanduser("~/ColabChem")
        if not os.path.isdir(os.path.join(path, "colabuma")):
            r = subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", ref,
                                "https://github.com/RChibaChiba/ColabChem.git", path], capture_output=True, text=True)
            if r.returncode != 0 or not os.path.isdir(os.path.join(path, "colabuma")):
                raise RuntimeError(f"Could not fetch the ColabChem shared code from GitHub (ref {ref}).\n{r.stderr}")
    if path not in sys.path:
        sys.path.insert(0, path)


_use_colabuma(COLABUMA_REF)

from colabuma import env, opt, output, scan, structure, ui
from colabuma.ui import ok

ui.install_error_handler()

fmax = opt.fmax_from_form(convergence, fmax_custom)
structure.check_form(fmax, max_steps, multiplicity)
job = opt.Job(structure.out_prefix(output_name), charge, multiplicity, model_name, task_name,
              optimizer_name, fmax, max_steps, convergence)
spec = scan.parse_spec(scan_atoms, scan_start, scan_n_steps, scan_step_size, opt_before_scan, opt_after_scan)
env.check_gpu()
env.ensure_token()
upload_name, upload_data = structure.upload()
scan.check_atom_range(spec, structure.quick_atom_count(upload_name, upload_data),
                      "Fix the form and run all again. In the file selection, Cancel upload reuses the previous file.")
ok("Input checked. You can leave it running from here.")

In [ ]:
#@title 3. Setup (install and login) { display-mode: "form" }
DEVICE = env.setup()

In [ ]:
#@title 4. Load the model { display-mode: "form" }
calc = env.load_calculator(model_name, task_name)

In [ ]:
#@title 5. Prepare the structures { display-mode: "form" }
atoms0, source_desc = structure.read_structure(upload_name, upload_data, charge, multiplicity)
ok(f"Initial structure ready: {atoms0.get_chemical_formula()} ({len(atoms0)} atoms).")
sc = scan.prepare(spec, atoms0)

In [ ]:
#@title 6. Scan { display-mode: "form" }
#@markdown The Converge bar shows how far fmax has come, in orders of magnitude, from its first value toward the threshold. It moves back if fmax rises.
scan_res = scan.run(sc, atoms0, calc, job)

In [ ]:
#@title 7. Results and save { display-mode: "form" }
files, log = scan.report(scan_res, job, atoms0)
files["log.txt"] = output.log_text("ColabUMA Scan log", source_desc, job, [log])
zip_path = output.save_zip(job.out_prefix, files)